In [1]:
import pandas as pd
import sqlalchemy as db

In [2]:
conn2 = db.create_engine('postgresql+psycopg2://postgres:<password>@localhost/keywords')

In [3]:
all_data_q = " SELECT name, abstract, icpsr_terms,  model_selected_terms_no_hal FROM icpsr_data_desc "

with conn2.connect() as conn :
    result = conn.execute(db.text(all_data_q))

all_datasets = pd.DataFrame(result.all(), columns=result.keys())

OperationalError: (psycopg2.OperationalError) connection to server at "localhost" (127.0.0.1), port 5432 failed: FATAL:  password authentication failed for user "postgres"
connection to server at "localhost" (127.0.0.1), port 5432 failed: FATAL:  password authentication failed for user "postgres"

(Background on this error at: https://sqlalche.me/e/20/e3q8)

In [13]:
all_datasets['name'] = all_datasets['name'].str.replace("'", "", regex=False)

print(all_datasets)

                                                  name  \
0    Longitudinal Evaluation of Chicagos Community ...   
1            National Mortality Followback Survey 1993   
2    Uniform Crime Reporting Program Data [United S...   
3    Enhanced Services for the Hard-to-Employ Cente...   
4    Arab Barometer: Public Opinion Survey Conducte...   
..                                                 ...   
495  Current Population Survey May 1999:  Tobacco U...   
496  Sierra Leone Threshold - Water Sector Reform: ...   
497  National Crime Victimization Survey 2008 [Coll...   
498  National Evaluation of the Safe Start Promisin...   
499  Federal Justice Statistics Program: Charges Fi...   

                                              abstract  \
0    The purpose of this study was to evaluate the ...   
1    The National Mortality Followback Survey (NMFS...   
2    These data provide information on the number o...   
3    The Enhanced Services for the Hard-to-Employ (...   
4    The Arab

In [5]:
ICPSR = pd.read_xml('subject_terms.xml')

ICPSR_list=ICPSR['DESCRIPTOR']
ICPSR_list=list(set(ICPSR_list))

In [17]:
import pandas as pd
from concurrent.futures import ThreadPoolExecutor
from sqlalchemy import text

def process_dataset(j):
    results = []

    for i in ICPSR_list:
        term = str(i)
        clean = (
            term.replace("'", "")
                .replace("(", "")
                .replace(")", "")
                .replace("&", "")
                .replace("|", "")
                .lstrip()
                .replace("  ", "")
                .replace(" ", "<->")
        )

        vctr_q = f"""
            SELECT name FROM icpsr_data_desc
            WHERE name = '{j}'
            AND search_vector @@ to_tsquery('{clean}');
        """

        with conn2.connect() as conn:
            result = conn.execute(text(vctr_q))
            df = pd.DataFrame(result.all(), columns=result.keys())

            if not df.empty:
                results.append(i)

    res2 = ' | '.join(str(x) for x in results if pd.notnull(x))
    return j, res2

In [19]:
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm import tqdm

results_map = {}
futures = []

with ThreadPoolExecutor(max_workers=5) as executor:
    for j in all_datasets['name']:
        futures.append(executor.submit(process_dataset, j))

    for future in tqdm(as_completed(futures), total=len(futures)):
        j, res2 = future.result()
        results_map[j] = res2


100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 500/500 [55:21<00:00,  6.64s/it]


In [20]:
all_datasets['database_selected_terms'] = all_datasets['name'].map(results_map)

In [24]:
all_datasets.to_csv("outputWithDBTerms.csv", index=False)
print(all_datasets.head())

                                                name  \
0  Longitudinal Evaluation of Chicagos Community ...   
1          National Mortality Followback Survey 1993   
2  Uniform Crime Reporting Program Data [United S...   
3  Enhanced Services for the Hard-to-Employ Cente...   
4  Arab Barometer: Public Opinion Survey Conducte...   

                                            abstract  \
0  The purpose of this study was to evaluate the ...   
1  The National Mortality Followback Survey (NMFS...   
2  These data provide information on the number o...   
3  The Enhanced Services for the Hard-to-Employ (...   
4  The Arab Barometer is a multicountry social su...   

                                         icpsr_terms  \
0  citizen participation  |  community involvemen...   
1  accidents  |  causes of death  |  death record...   
2  arrest records  |  arrests  |  crime rates  | ...   
3  child support  |  conviction records  |  crimi...   
4  Arab Israeli conflict  |  democracy  |  eco